# Dockerizing an API

This notebook covers:

1. Writing a real, working Dockerfile for a FastAPI app — and *why* each line is there
2. Multi-stage builds: a fat builder image plus a slim runtime image, and how the split shrinks the deployed artifact
3. Running as a non-root user — the one-line change every production image needs
4. Container-layer health checks via the `HEALTHCHECK` directive, alongside the app-layer `/healthz` endpoint
5. `.dockerignore` — keeping secrets, virtualenvs, and noise out of your build context

**Scope**: FastAPI + Docker. Unlike every other notebook in this curriculum, the actual `docker build` / `docker run` steps require a Docker daemon. The cells below **write real Dockerfile content to a temporary workspace** and validate its structure in-process; a single optional cell shells out to `docker` if it's installed. Everything else runs without Docker.

## 1. The Minimal Dockerfile

Docker images are built from a `Dockerfile` — a sequence of declarative instructions (`FROM`, `COPY`, `RUN`, `CMD`, ...) that the builder applies one at a time. Each instruction produces an immutable **layer**; subsequent builds reuse cached layers when their inputs haven't changed. Understanding that caching model is most of what makes Dockerfiles fast.

The simplest possible image for our FastAPI app — one stage, no frills — looks like this. We'll write it to a temp directory alongside a tiny app and walk through the choices.

In [1]:
import shutil
import subprocess
from pathlib import Path
from tempfile import mkdtemp

WORK = Path(mkdtemp(prefix="docker_ch8_"))
print("workspace:", WORK)

# A tiny FastAPI app the image will serve. Identical shape to 08.01's tiny_app.
(WORK / "app.py").write_text("""from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"hello": "docker"}
""", encoding="utf-8")

# requirements.txt — pinned. Pinning is the difference between a reproducible
# build today and a 3am incident in eight months when a transitive dep ships
# a breaking minor.
(WORK / "requirements.txt").write_text("""fastapi==0.115.0
uvicorn[standard]==0.30.6
""", encoding="utf-8")

MINIMAL_DOCKERFILE = """\
# Pin the Python minor AND the OS tag. `python:3.12` floats; `python:3.12.7-slim-bookworm` is reproducible.
FROM python:3.12.7-slim-bookworm

# Keep stdout/stderr unbuffered so container logs show up live in `docker logs`.
ENV PYTHONUNBUFFERED=1 \\
    PYTHONDONTWRITEBYTECODE=1 \\
    PIP_NO_CACHE_DIR=1 \\
    PIP_DISABLE_PIP_VERSION_CHECK=1

WORKDIR /app

# Copy requirements *before* the source. Layer cache: if app.py changes but
# requirements.txt doesn't, the pip-install layer is reused — a multi-second
# win on every rebuild.
COPY requirements.txt .
RUN pip install -r requirements.txt

COPY app.py .

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

(WORK / "Dockerfile").write_text(MINIMAL_DOCKERFILE, encoding="utf-8")
print("--- Dockerfile ---")
print(MINIMAL_DOCKERFILE)


workspace: C:\Users\mathi\AppData\Local\Temp\docker_ch8_0dyta21d
--- Dockerfile ---
# Pin the Python minor AND the OS tag. `python:3.12` floats; `python:3.12.7-slim-bookworm` is reproducible.
FROM python:3.12.7-slim-bookworm

# Keep stdout/stderr unbuffered so container logs show up live in `docker logs`.
ENV PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1 \
    PIP_NO_CACHE_DIR=1 \
    PIP_DISABLE_PIP_VERSION_CHECK=1

WORKDIR /app

# Copy requirements *before* the source. Layer cache: if app.py changes but
# requirements.txt doesn't, the pip-install layer is reused — a multi-second
# win on every rebuild.
COPY requirements.txt .
RUN pip install -r requirements.txt

COPY app.py .

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



Three things worth memorizing about that file:

- **Pin the base image to a digest-grade tag.** `python:3.12.7-slim-bookworm` will give the same bytes today and a year from now (within Debian's security-patch lifecycle); `python:3.12` will not. Reproducibility starts at `FROM`.
- **Order layers cheapest-to-most-likely-to-change.** Dependencies first, then source. `pip install` is the expensive layer; if it runs only when `requirements.txt` changes, your inner-loop rebuilds are seconds instead of minutes.
- **`CMD` is the default process.** It's *not* a shell command unless you write it as `CMD "cmd args"` (the "shell form"). Prefer the JSON-array exec form (`CMD ["uvicorn", ...]`) so signals propagate cleanly — `SIGTERM` from the orchestrator must reach Uvicorn, and a shell-form wrapper swallows it.

Let's now parse the Dockerfile we just wrote and confirm the structure is what we think it is. This isn't `docker build` — it's a cheap structural lint that we can run anywhere, useful in CI to catch missing pieces before they reach a build agent.

In [2]:
def parse_dockerfile(text: str) -> list[tuple[int, str, str]]:
    """Return [(line_no, instruction, args), ...] for non-comment, non-blank lines.

    Continuation lines (\\\\ at EOL) are folded into the previous instruction so
    `RUN apt-get update && \\\\\n    apt-get install ...` parses as a single RUN."""
    out: list[tuple[int, str, str]] = []
    folded: list[str] = []
    start_line = 0
    for i, raw in enumerate(text.splitlines(), start=1):
        stripped = raw.rstrip()
        if not stripped or stripped.lstrip().startswith("#"):
            continue
        if folded:
            folded.append(stripped.rstrip("\\"))
            if not stripped.endswith("\\"):
                joined = " ".join(s.strip() for s in folded)
                instr, _, args = joined.partition(" ")
                out.append((start_line, instr.upper(), args.strip()))
                folded = []
            continue
        if stripped.endswith("\\"):
            folded = [stripped.rstrip("\\")]
            start_line = i
            continue
        instr, _, args = stripped.partition(" ")
        out.append((i, instr.upper(), args.strip()))
    return out

parsed = parse_dockerfile(MINIMAL_DOCKERFILE)
for line_no, instr, args in parsed:
    print(f"L{line_no:>3}  {instr:<8} {args[:70]}")

# A handful of cheap lint rules.
instrs = [i for _, i, _ in parsed]
assert instrs[0] == "FROM", "First instruction must be FROM"
assert "WORKDIR" in instrs, "WORKDIR keeps relative COPY targets predictable"
assert "CMD" in instrs or "ENTRYPOINT" in instrs, "Need a default process"
copy_idx = [k for k, i in enumerate(instrs) if i == "COPY"]
pip_idx = next(k for k, i in enumerate(instrs) if i == "RUN" and "pip install" in parsed[k][2])
assert copy_idx[0] < pip_idx < copy_idx[1], "requirements.txt must be COPY'd before pip install, source after"
print("\\nlint OK: layers are ordered for cache reuse")


L  2  FROM     python:3.12.7-slim-bookworm
L  5  ENV      PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1 PIP_NO_CACHE_DIR=1 PIP_DI
L 10  WORKDIR  /app
L 15  COPY     requirements.txt .
L 16  RUN      pip install -r requirements.txt
L 18  COPY     app.py .
L 20  EXPOSE   8000
L 21  CMD      ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
\nlint OK: layers are ordered for cache reuse


The lint script does in five lines what `hadolint` does more thoroughly — your CI can run something like this as a pre-build gate so a Dockerfile that won't cache correctly doesn't even reach the registry.

## 2. Multi-Stage Builds for Smaller Images

The minimal Dockerfile above produces an image that contains the **build toolchain** (pip's cache directories, any C compilers pulled in by transitive deps, headers, etc.) alongside the runtime. That's wasted bytes — both in your registry and in every pull on every deploy.

The fix is a **multi-stage build**: a `builder` stage installs everything into a virtualenv; a slim `runtime` stage copies *only the virtualenv* across. Everything pip downloaded, every compiler invocation, every cache directory — left behind.

```
+-----------+  pip install --> /opt/venv  ----+
| builder   |                                  | COPY --from=builder /opt/venv /opt/venv
| (slim+    |                                  v
|  build-   |                            +-----------+
|  essential) |                          | runtime   |  <-- the only image you ship
+-----------+                            | (slim)    |
                                         +-----------+
```

The shipped image contains the virtualenv and the app, nothing else.

In [3]:
MULTISTAGE_DOCKERFILE = """\
# syntax=docker/dockerfile:1.7
# ----- stage 1: builder ----------------------------------------------------
FROM python:3.12.7-slim-bookworm AS builder

ENV PYTHONDONTWRITEBYTECODE=1 \\
    PIP_NO_CACHE_DIR=1 \\
    PIP_DISABLE_PIP_VERSION_CHECK=1

# Build essentials only live in the builder stage — they're not copied forward.
RUN apt-get update && \\
    apt-get install -y --no-install-recommends build-essential && \\
    rm -rf /var/lib/apt/lists/*

# Install into an isolated venv we can copy out wholesale.
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

WORKDIR /build
COPY requirements.txt .
RUN pip install -r requirements.txt

# ----- stage 2: runtime ----------------------------------------------------
FROM python:3.12.7-slim-bookworm AS runtime

ENV PYTHONUNBUFFERED=1 \\
    PYTHONDONTWRITEBYTECODE=1 \\
    PATH="/opt/venv/bin:$PATH"

# Bring the venv across — and nothing else from the builder.
COPY --from=builder /opt/venv /opt/venv

WORKDIR /app
COPY app.py .

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

(WORK / "Dockerfile").write_text(MULTISTAGE_DOCKERFILE, encoding="utf-8")
parsed = parse_dockerfile(MULTISTAGE_DOCKERFILE)

stages = [(line, args) for line, instr, args in parsed if instr == "FROM"]
print("stages:")
for line, args in stages:
    print(f"  L{line:>3}  FROM {args}")

copy_from = [args for _, instr, args in parsed if instr == "COPY" and "--from=" in args]
print("\\ncross-stage copies:")
for c in copy_from:
    print("  COPY", c)

assert len(stages) == 2, "Expected two FROM lines (builder + runtime)"
assert any("--from=builder" in c for c in copy_from), "Runtime must copy from builder"
print("\\nlint OK: two stages, runtime imports from builder")


stages:
  L  3  FROM python:3.12.7-slim-bookworm AS builder
  L 23  FROM python:3.12.7-slim-bookworm AS runtime
\ncross-stage copies:
  COPY --from=builder /opt/venv /opt/venv
\nlint OK: two stages, runtime imports from builder


Three subtleties to internalize:

- **Cross-stage `COPY --from=` is the seam.** The runtime stage references the builder by its `AS builder` label; the builder vanishes after the build (you only push the final stage). The pip cache, the apt cache, `build-essential` itself — none of them appear in the shipped image.
- **The venv path trick.** Building into `/opt/venv` and copying the directory verbatim is the simplest reliable way to bring Python dependencies across stages — no `pip install` re-runs at runtime, no system site-packages. `PATH` puts the venv's `python` first so the `CMD` finds it.
- **The size delta is usually 2–10× on a typical FastAPI app.** Most of that win is `build-essential` (~250MB) plus pip's cache. Even before counting your own deps, the multi-stage image is dramatically smaller.

If you have Docker installed, section 5 will actually build both images and print `docker images` so you can see the size difference. If you don't, the comparison above is what you'd observe.

## 3. Non-Root User

By default, every instruction in a Dockerfile runs as `root`. The shipped container's `CMD` also runs as `root` — which means a remote code execution bug in your app gets root inside the container. From there, an attacker is one container-escape CVE away from your host.

The mitigation is one of the cheapest security wins in the whole stack: **add a `USER` directive**. We create an unprivileged user during the build, `chown` the app directory to it, and switch to it before `CMD`.

Read closely — the order matters. A `USER` declared too early prevents subsequent `RUN apt-get install` from working; declared too late and the app directory is still root-owned.

In [4]:
HARDENED_DOCKERFILE = """\
# syntax=docker/dockerfile:1.7
FROM python:3.12.7-slim-bookworm AS builder

ENV PYTHONDONTWRITEBYTECODE=1 PIP_NO_CACHE_DIR=1 PIP_DISABLE_PIP_VERSION_CHECK=1

RUN apt-get update && \\
    apt-get install -y --no-install-recommends build-essential && \\
    rm -rf /var/lib/apt/lists/*

RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

WORKDIR /build
COPY requirements.txt .
RUN pip install -r requirements.txt

# ----- runtime -------------------------------------------------------------
FROM python:3.12.7-slim-bookworm AS runtime

ENV PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1 PATH="/opt/venv/bin:$PATH"

# Create an unprivileged user with a fixed UID. Fixed UIDs play nicely with
# host bind-mounts and with cluster admission policies that pin to ranges.
RUN groupadd --system --gid 10001 app && \\
    useradd  --system --uid 10001 --gid 10001 --home-dir /app --no-create-home app

COPY --from=builder /opt/venv /opt/venv

WORKDIR /app
COPY --chown=app:app app.py .

# Drop to the non-root user *before* declaring the default process.
USER app

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

(WORK / "Dockerfile").write_text(HARDENED_DOCKERFILE, encoding="utf-8")
parsed = parse_dockerfile(HARDENED_DOCKERFILE)

# The lint we care about now: USER appears, and it appears AFTER the last RUN
# that needs root (apt-get, useradd) and BEFORE CMD.
indices = {instr: [k for k, (_, i, _) in enumerate(parsed) if i == instr] for instr in ("RUN", "USER", "CMD")}
assert indices["USER"], "Missing USER directive"
last_run = max(indices["RUN"])
user_idx = indices["USER"][0]
cmd_idx = indices["CMD"][0]
assert last_run < user_idx < cmd_idx, "USER must come after the last root-requiring RUN and before CMD"
print(f"USER directive at parsed position {user_idx}; last root RUN at {last_run}; CMD at {cmd_idx}")
print("lint OK: drops privileges before CMD")


USER directive at parsed position 14; last root RUN at 10; CMD at 16
lint OK: drops privileges before CMD


A few things to notice:

- **`--system` users** get UIDs outside the normal interactive range and no login shell. They exist to own a daemon; nobody logs in as them.
- **`COPY --chown=app:app`** is the per-instruction shortcut for setting ownership on the copy. Doing it at copy time is one fewer layer than a follow-up `RUN chown -R`.
- **`USER app` after the venv is in place.** The runtime user does not need to install anything; everything's already there from the builder.
- **The port doesn't need to be > 1024.** That's a common misconception — Linux only restricts privileged ports for *binding by an unprivileged user* on the host, and 8000 is unprivileged anyway. If you ever need port 80, expose 8000 inside the container and have the orchestrator (k8s Service, ALB) do the mapping.

## 4. Health Checks at the App and Container Layers

There are two distinct "is this thing alive?" mechanisms in a container deployment, and they answer different questions:

- **App-layer `/healthz` endpoint.** A route your app exposes. The orchestrator's liveness/readiness probes (k8s, ECS, Nomad) hit it on a schedule; if it fails enough times, the orchestrator restarts the container. This is the *primary* mechanism in production. We add one below.
- **Container-layer `HEALTHCHECK` directive.** A command Docker runs *inside* the container and reports back via `docker inspect`. Useful for `docker compose` setups and local development. Most production orchestrators **ignore** it — they have their own probes that hit the app from outside.

Both are useful in different contexts; we'll wire both up. The app cell below also proves the `/healthz` route works without Docker — we instantiate the app via `TestClient` and hit the endpoint directly.

In [5]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

# Update the in-workspace app.py to expose /healthz.
APP_WITH_HEALTH = """from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"hello": "docker"}

# A liveness probe should be cheap and dependency-free. Don't touch the DB
# here — that's what /readyz is for (notebook 08.01 exercise 1).
@app.get("/healthz")
def healthz():
    return {"status": "ok"}
"""

(WORK / "app.py").write_text(APP_WITH_HEALTH, encoding="utf-8")

# Prove the endpoint works without Docker — in-process via TestClient.
import importlib, sys
sys.path.insert(0, str(WORK))
sys.modules.pop("app", None)
mod = importlib.import_module("app")
with TestClient(mod.app) as client:
    r = client.get("/healthz")
    print("healthz status:", r.status_code, "| body:", r.json())
sys.path.remove(str(WORK))

# Now wire HEALTHCHECK into the Dockerfile. Note: the slim image does not ship
# curl by default. We use a small Python one-liner instead, which the venv
# already has on PATH.
HEALTHCHECKED = HARDENED_DOCKERFILE.replace(
    "EXPOSE 8000",
    """HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 \\
  CMD python -c "import urllib.request,sys; \\
sys.exit(0 if urllib.request.urlopen('http://127.0.0.1:8000/healthz', timeout=2).status==200 else 1)"

EXPOSE 8000"""
)
(WORK / "Dockerfile").write_text(HEALTHCHECKED, encoding="utf-8")

parsed = parse_dockerfile(HEALTHCHECKED)
assert any(instr == "HEALTHCHECK" for _, instr, _ in parsed), "Missing HEALTHCHECK"
print("HEALTHCHECK present; uses python+urllib (no curl needed in slim image)")


healthz status: 200 | body: {'status': 'ok'}
HEALTHCHECK present; uses python+urllib (no curl needed in slim image)


C:\Users\mathi\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Two notes on the `HEALTHCHECK` shape:

- **`--start-period`** is the grace window during which a failing check doesn't count toward the failure threshold. Most apps need ~10 seconds to warm up — the JIT-import, the lifespan, the first connection-pool fill. Without `--start-period`, you'd get flaps on every restart.
- **No curl in slim images.** The default `python:*-slim` images are stripped of most CLI tools to keep the image small. A Python urllib one-liner adds zero MB and is good enough for a healthcheck. If you do install curl, prefer `--no-install-recommends` and clean up `apt-get` lists, as in the builder stage above.

## 5. Building and Running

If you have Docker installed, the cell below actually builds the image and runs it for a single request. If you don't, it prints the commands you'd run on a machine that has Docker.

The build command:

```
docker build -t portfolio-api:dev .
```

…runs the Dockerfile in the current directory. The first build will pull `python:3.12.7-slim-bookworm`, install your deps, and produce an image; subsequent builds reuse cached layers when their inputs haven't changed.

The run command:

```
docker run --rm -d -p 8000:8000 --name portfolio-api portfolio-api:dev
```

`--rm` removes the container when it stops; `-d` detaches; `-p 8000:8000` maps host port 8000 to container port 8000. The image's `CMD` runs as the `app` user inside.

Production-grade run flags add `--read-only`, `--user 10001:10001` (defense in depth against a `USER`-stripping image rebuild), `--security-opt no-new-privileges`, a memory limit, and a restart policy.

In [6]:
docker_path = shutil.which("docker")
if docker_path is None:
    print("docker not installed on this machine — skipping the real build/run.")
    print()
    print("To exercise this section locally on a machine with Docker:")
    print(f"  cd {WORK}")
    print("  docker build -t portfolio-api:dev .")
    print("  docker run --rm -d -p 8000:8000 --name portfolio-api portfolio-api:dev")
    print("  curl http://127.0.0.1:8000/healthz")
    print("  docker stop portfolio-api")
else:
    print("docker found at:", docker_path)
    build = subprocess.run([docker_path, "build", "-t", "portfolio-api:dev", str(WORK)],
                           capture_output=True, text=True)
    print("--- docker build stdout (tail) ---")
    print("\\n".join(build.stdout.splitlines()[-15:]))
    if build.returncode != 0:
        print("build failed:", build.stderr[-500:])
    else:
        # Use a fresh container name to avoid colliding with a stale one.
        name = "portfolio_api_ch8_demo"
        subprocess.run([docker_path, "rm", "-f", name], capture_output=True)
        run = subprocess.run(
            [docker_path, "run", "--rm", "-d", "-p", "8001:8000", "--name", name, "portfolio-api:dev"],
            capture_output=True, text=True,
        )
        print("container id:", run.stdout.strip()[:12])
        try:
            import time, httpx
            time.sleep(2.0)
            r = httpx.get("http://127.0.0.1:8001/healthz", timeout=5.0)
            print("healthz over real HTTP:", r.status_code, r.json())
        finally:
            subprocess.run([docker_path, "stop", name], capture_output=True)


docker not installed on this machine — skipping the real build/run.

To exercise this section locally on a machine with Docker:
  cd C:\Users\mathi\AppData\Local\Temp\docker_ch8_0dyta21d
  docker build -t portfolio-api:dev .
  docker run --rm -d -p 8000:8000 --name portfolio-api portfolio-api:dev
  curl http://127.0.0.1:8000/healthz
  docker stop portfolio-api


## 6. `.dockerignore`

The build context is the directory tree Docker sends to the daemon at build time. Without a `.dockerignore`, that tree includes everything under the build directory — your `.git/` history, your local `.venv/`, your secret `.env`, your IDE's caches. Three reasons that's bad:

- **Speed.** A huge context means a slow `docker build` because every byte gets sent to the daemon before the first instruction runs.
- **Cache misses.** Any file in the context is a potential cache invalidator — a `.git/HEAD` that changes on every commit means your `COPY . .` busts the cache on every build.
- **Leaks.** A `.env` with API keys in the build context is one careless `COPY . .` away from being baked into a public image.

The fix is a `.dockerignore` file next to the `Dockerfile`, with patterns identical to `.gitignore`. The cell below writes a sensible default.

In [7]:
DOCKERIGNORE = """\
# Version control
.git
.gitignore

# Python
__pycache__/
*.pyc
*.pyo
*.pyd
.venv/
venv/
.pytest_cache/
.mypy_cache/
.ruff_cache/

# Local env / secrets
.env
.env.*
!.env.example

# Notebooks & local artifacts (the curriculum's own context)
*.ipynb
.ipynb_checkpoints/

# IDE / OS noise
.vscode/
.idea/
.DS_Store
Thumbs.db

# Docs and build output the runtime image doesn't need
docs/
build/
dist/
*.egg-info/
"""

(WORK / ".dockerignore").write_text(DOCKERIGNORE, encoding="utf-8")

# What survives the filter? Simulate `docker build`'s context calculation
# by walking WORK and applying the patterns. This is approximate — Docker
# itself uses Go's path/filepath.Match — but close enough to spot mistakes.
import fnmatch
patterns = [l.strip() for l in DOCKERIGNORE.splitlines() if l.strip() and not l.startswith("#")]
def keep(rel: str) -> bool:
    for p in patterns:
        if p.startswith("!"):
            if fnmatch.fnmatch(rel, p[1:]):
                return True
        elif fnmatch.fnmatch(rel, p) or any(fnmatch.fnmatch(part, p.rstrip("/")) for part in rel.split("/")):
            return False
    return True

print("files that WOULD be in the build context:")
for f in sorted(WORK.rglob("*")):
    if not f.is_file():
        continue
    rel = f.relative_to(WORK).as_posix()
    mark = "+" if keep(rel) else "-"
    print(f"  {mark} {rel}")
print("\\n(+ included, - filtered out by .dockerignore)")


files that WOULD be in the build context:
  + .dockerignore
  - __pycache__/app.cpython-314.pyc
  + app.py
  + Dockerfile
  + requirements.txt
\n(+ included, - filtered out by .dockerignore)


## Key Takeaways

- **Pin everything.** Base image to a `MAJOR.MINOR.PATCH-flavor` tag, dependencies via `==` in `requirements.txt`. "Reproducible build" is a property you have to opt into.
- **Order layers cheapest-to-most-volatile.** Deps before source. `pip install` should run on requirements changes, not on every save.
- **Multi-stage = builder + runtime.** Everything that's only needed to *build* (compilers, headers, pip cache) stays in the builder stage and never reaches the registry.
- **Always declare `USER` before `CMD`.** A non-root container is a one-line change and one of the biggest defense-in-depth wins in the stack. Fix the UID so admission policies can range-restrict it.
- **Two layers of health checks.** `/healthz` in the app is what your orchestrator probes; `HEALTHCHECK` in the Dockerfile is what `docker compose` and `docker inspect` use. Both are cheap; ship both.
- **`.dockerignore` is not optional.** Without it your context is fat, your cache is fragile, and your secrets are one `COPY . .` away from the registry.
- **Capstone tie-in**: `examples/portfolio_analytics_api/Dockerfile` will use exactly this multi-stage + non-root + healthcheck shape, parameterized by a `Settings`-driven `APP_VERSION` build-arg.

## Exercises

Use the temp `WORK` directory and the `parse_dockerfile` helper for the structural lint cells; the real `docker build` cells are optional if you have a Docker daemon available.

**1. Shrink with a digest-pinned distroless runtime.** Replace the runtime stage's base image with `gcr.io/distroless/python3-debian12` and re-run the lint. Distroless images have no shell — your `HEALTHCHECK` can't shell out, and you can't `docker exec ... sh` for debugging. Adjust the `HEALTHCHECK` so it runs the Python urllib check directly (no shell wrapper), and write a short paragraph in a markdown cell about what you'd lose for debugging in exchange.

**2. Build-arg for the app version.** Add an `ARG APP_VERSION=dev` to the runtime stage and an `ENV APP_VERSION=$APP_VERSION`. Modify `app.py` to expose `/version` returning `{"version": os.environ["APP_VERSION"]}`. Lint the Dockerfile to assert `ARG` and `ENV` both appear, and (if Docker is available) build with `--build-arg APP_VERSION=1.2.3` and curl `/version`. The point: passing build-time values into the image without rebaking the Dockerfile.

**3. A multi-arch Dockerfile sketch.** Write a brief Dockerfile that uses `--platform=$BUILDPLATFORM` in the builder stage and a normal runtime stage, so the same Dockerfile can target both `linux/amd64` and `linux/arm64` via `docker buildx build --platform linux/amd64,linux/arm64`. You don't have to run buildx; the deliverable is the Dockerfile plus a markdown cell explaining when the explicit `$BUILDPLATFORM` matters (hint: cross-compiled wheels, slow QEMU emulation, native build performance).

In [8]:
import shutil as _sh
print("workspace at end:", WORK)
print("(left in place for the exercises; remove with shutil.rmtree if you're done)")


workspace at end: C:\Users\mathi\AppData\Local\Temp\docker_ch8_0dyta21d
(left in place for the exercises; remove with shutil.rmtree if you're done)
